<a href="https://colab.research.google.com/github/BakrAdli/My_AI_Journey/blob/main/smart_car_diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# V2.9 : Multi-Car Garage Fleet Baseline (V3.0 LLM Ready)
import json
import os
from typing import Dict, List, Optional, Union, Any

def clear_screen() -> None:
    """Colab-friendly separator to prevent input box glitching"""
    print("\n"+"="*55+"\n")


def get_valid_integer(prompt: str, min_val: int = 0) -> int:
    """Helper function to cleanly handel integer inputs"""
    while True:
        try:
            val = int(input(prompt).strip())
            if val < min_val:
                print(f"  [ERROR] Value cannot be less than {min_val}")
                continue
            return val
        except ValueError :
            print("  [ERROR] Invalid input. Numbers only. ")

class Car:
    def __init__(
        self,
        brand: str,
        model: str,
        engine_type: str,
        mileage: int,
        last_oil_change: int = 0,
        last_tire_change: int = 0,
        trip_history: Optional[List[int]] = None
    ) -> None:
        self.brand = brand
        self.model = model
        self.engine_type = engine_type
        self.mileage = mileage
        self.last_oil_change = last_oil_change
        self.last_tire_change = last_tire_change
        self.trip_history = trip_history if trip_history is not None else []

    def add_trip(self, distance: int) -> bool:
        if distance <= 0:
            print("\n  [Error] Distance must be > 0. Cannot add negative/zero km.")
            return False

        self.mileage += distance
        self.trip_history.append(distance)

        if len(self.trip_history) > 50:
            self.trip_history = self.trip_history[-50:]

        print(f"\n  [System] Trip of {distance:,} km recorded! Odometer: {self.mileage:,} km")
        return True

    def reset_service(self, service_type: str) -> None:
        """Unified method for resetting services efficiently."""
        if service_type == "oil":
            self.last_oil_change = self.mileage
            print(f"\n  [Service] Oil reset at {self.mileage:,} km!")
        elif service_type == "tires":
            self.last_tire_change = self.mileage
            print(f"\n  [Service] Tires reset at {self.mileage:,} km!")

    def edit_data(self) -> None:
        print("\n[*] Edit Vehicle Details (Press Enter to keep current value): ")

        new_brand = input(f" Brand [{self.brand}] : ").strip()
        if new_brand: self.brand = new_brand

        new_model = input(f" Model [{self.model}] : ").strip()
        if new_model: self.model = new_model

        new_engine = input(f" Engine [{self.engine_type}] : ").strip()
        if new_engine: self.engine_type = new_engine

        while True:
            new_mile = input(f" Mileage [{self.mileage}] : ").strip()
            if not new_mile:
                break
            try:
                temp_mileage = int(new_mile)
                if temp_mileage < self.mileage:
                    print(f"   [Error] Fraud Alert! New mileage ({temp_mileage:,}) < current ({self.mileage:,})!")
                    continue
                self.mileage = temp_mileage
                break
            except ValueError:
                print("   [Error] Invalid input. Positive integers only.")

        self.save_data(silent=True)
        print("  [System] Data updated successfully!")

    def get_oil_status(self) -> str:
        km_driven = self.mileage - self.last_oil_change
        if km_driven >= 10000:
            return f"CHANGE_REQUIRED ({km_driven:,} km driven)"
        return f"OK ({10000 - km_driven:,} km left)"

    def get_tires_status(self) -> str:
        km_driven = self.mileage - self.last_tire_change
        if km_driven >= 50000:
            return f"CHECK_TREAD ({km_driven:,} km driven)"
        return f"OK ({50000 - km_driven:,} km left)"

    def get_engine_tip(self) -> str:
        tips = {
            "GASOLINE": "Check spark plugs & fuel injectors",
            "DIESEL": "Check Glow Plugs & EGR Valve",
            "HYBRID": "Schedule battery health diagnostic",
            "ELECTRIC": "Inspect thermal management & brake regen"
        }
        return tips.get(self.engine_type, "Standard engine maintenance")

    def get_llm_context(self) -> Dict[str, Any]:
        """Returns vehicle context dictionary ready for JSON serialization"""
        return  {
            "vehicle": f"{self.brand} {self.model}",
            "mileage": self.mileage,
            "history": {
                "oil_change_at": self.last_oil_change,
                "tire_change_at": self.last_tire_change,
                "recent_trips": self.trip_history[-5:]
            },
            "diagnostics": {
                "oil": self.get_oil_status(),
                "tires": self.get_tires_status(),
                "engine": self.get_engine_tip()
            }
        }
        return json.dumps(context, indent=4)

    def display_dashboard(self) -> None:
        oil_stat = self.get_oil_status()
        tire_stat = self.get_tires_status()
        engine_stat = self.get_engine_analysis()

        print("\n" + "="*50)
        print("  SYSTEM DIAGNOSTICS & LOGBOOK ")
        print("="*50)
        print(f" VEHICLE : {self.brand.upper()} {self.model.upper()}")
        print(f" MILEAGE : {self.mileage:,} (km) | TRIPS: {len(self.trip_history)}")
        print("-" * 50)
        print(f" OIL     : [{'' if 'CHANGE' in oil_stat else ''}] {oil_stat}")
        print(f" TIRES   : [{'' if 'CHECK' in tire_stat else ''}] {tire_stat}")
        print(f" ENGINE  :  {engine_stat['type']} ->  {engine_stat['tip']}")
        print("="*50)

    def to_dict(self) -> Dict[str, Any]:
        """Serialize car object for JSON storage"""
        return  {
            "brand": self.brand,
            "model": self.model,
            "engine_type": self.engine_type,
            "mileage": self.mileage,
            "last_oil_change": self.last_oil_change,
            "last_tire_change": self.last_tire_change,
            "trip_history": self.trip_history
        }
    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> 'Car':
        """Instantiates a Car object from dictionary data"""
        return cls(
            brand=data.get("brand", "Unknown"),
            model=data.get("model", "Unknown"),
            engine_type=data.get("engine_type", "Unknown"),
            mileage=data.get("mileage",0),
            last_oil_change=data.get("last_oil_change", 0),
            last_tire_change=data.get("last_tires_change", 0),
            trip_history=data.get("trip_history", [])
        )
#======== {CLASS [2] : Garage }==========
class Garage:
    def __init__(self, storage_file: str = "fleet_data.json") -> None:
        self.storage_file = storage_file
        self.cars: List[Car] = []
        self.active_index: int = 0
        self.load_fleet()
    @property
    def active_car(self) -> Optional[Car]:
        if 0 <= self.active_index < len(self.cars):
            return self.cars[self.active_index]
        return None

    def add_car(self, car: Car) -> None:
        self.cars.append(car)
        self.active_index = len(self.cars) -1 # Auto-Switch to newly added car
        self.save_fleet()
        print(f"\n [Garage] added {car.brand}{car.model} to fleet and set as Active ! ")
    def ban_car(self, index: int) -> bool:
        if len(self.cars) <= 1:
            print("\n [ERROR] Cannot ban/remove the only reamining car in your garage ! ")
            return False
        if 0 <= index < len(self.cars):
            banned_car = self.cars.pop(index)
            print(f"\n [Garage] Banned {banned_car.brand} {banned_car.model} from fleet")
            # Adjust active_index safely
            if self.active_index >= len(self.cars):
                self.active_index = len(self.cars) -1
            elif self.active_index == index:
                self.active_index = 0

            self.save_fleet()
            return True
        else:
            print("\n [ERROR] Invalid car selection index")
            return False

    def switch_car(self, index: int) -> bool:
        if 0 <= index < len(self.cars):
            self.active_index = index
            self.save_fleet()
            print(f"\n [Garage] Switched active vehicle to: {self.active_car.brand} {self.active_car.model}")
            return True
        print("\n [ERROR] Invalid car selection index")
        return False

    def get_fleet_llm_context(self) -> str:
        """Baseline LLM context aggregator for V3.0 Integration"""
        fleet_context = {
            "garage_summary":{
                "total_vehicles": len(self.cars),
                "active_vehicle_index": self.active_index,
            },
            "fleet_vehicles": [
                {
                    "garage_id": idx,
                    "is_active_vehicle": (idx == self.active_index),
                    "car_data": car.get_llm_context()
                }
                for idx, car in enumerate(self.cars)
            ]
        }
        return json.dumps(fleet_context, indent=4)

    def save_fleet(self) -> None:
        data = {
            "active_index": self.active_index,
            "cars": [car.to_dict() for car in self.cars]
        }
        with open(self.storage_file, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4)
            print(f"   [Storage] Saved to '{self.storage_file}'. (Click ' Refresh' in Colab if hidden)")

    def load_fleet(self) -> None:

        if os.path.exists(self.storage_file):
            try:
                with open(self.storage_file, "r", encoding="utf-8") as file:
                    data = json.load(file)
                    self.active_index = data.get("active_index", 0)
                    self.cars = [Car.from_dict(c) for c in data.get("cars", [])]
            except Exception:
                self.cars = []
        elif os.path.exists("car_data.json"):
            try:
                with open("car_data.json", "r", encoding="utf-8") as file:
                    old_data = json.load(file)
                old_car = Car.from_dict(old_data)
                self.cars = [old_car]
                self.active_index = 0
                self.save_fleet()
                print(" [Migration] v2.7 car_data.json imported into v2.8 Garage fleet ! ")
            except Exception:
                self.cars = []


    def print_active_banner(self) -> None:
        """V2.9 Feature: Prints active vehicle banner."""
        if not self.cars:
            return
        car = self.active_car
        print("\n" + "═"*55)
        print(f"   ACTIVE VEHICLE: {car.brand} {car.model}")
        print(f"  • Engine : {car.engine_type} | Mileage: {car.mileage:,} km")
        print(f"  • Status : Oil [{car.get_oil_status()}] | Tires [{car.get_tires_status()}]")
        print("═"*55)

    def show_maintenance_dashboard(self) -> None:
        """V2.9 Feature: Fleet Maintenance Overview."""
        print("\n" + "═"*55)
        print("  FLEET MAINTENANCE DASHBOARD (HEALTH OVERVIEW)")
        print("═"*55)
        for idx, car in enumerate(self.cars):
            tag = " [ACTIVE]" if idx == self.active_index else ""
            print(f"\n [{idx}] {car.brand} {car.model} ({car.mileage:,} km){tag}")
            print(f"   ├─ Oil Status  : {car.get_oil_status()}")
            print(f"   ├─ Tire Status : {car.get_tires_status()}")
            print(f"   └─ Engine Tip  : {car.get_engine_tip()}")
        print("═"*55)

# ========= MAIN APPLICATION LOOP ==========

if __name__ == "__main__":
    clear_screen()
    print("== Smart Diagnostics Initialized ==\n")

    garage = Garage()

    # Initial registration if garage is empty
    if not garage.cars:
        print("[*] No vehicles found in Garage. Create your first vehicle : ")
        u_brand = input(" Brand  : ").strip()
        u_model = input(" Model  : ").strip()
        u_engine = input(" Engine : ").strip()
        u_mileage = get_valid_integer(" Mileage: ")

        first_car = Car(u_brand, u_model, u_engine, u_mileage)
        garage.add_car(first_car)

    # Interactive Loop

    while True:
        clear_screen()
        garage.print_active_banner()
        print("\n" + "="*50)
        print(f" GARAGE MANAGMENT SYSTEM V2.8 (Fleet: {len(garage.cars)})")
        print("="*50)

        print("\n[Menu Options]:")
        print(" 1. Add Trip         ")
        print(" 2. Reset Oil        ")
        print(" 3. Reset Tires      ")
        print(" 4. Edit Vehicle     ")
        print(" 5. Garage Fleet Manager")
        print(" 6. Fleet Health Dashboard (NEW)")
        print(" 7. View AI Context  ")
        print(" 8. Exit             ")

        choice = input("\nSelect (1-8): ").strip()

        if choice == '1':
            dist = get_valid_integer("\nTrip distance (KM): ", min_val=1)
            if garage.active_car.add_trip(dist):
                garage.save_fleet()
                input("\nPress Enter...")

        elif choice == '2':
            garage.active_car.reset_service("oil")
            garage.save_fleet()
            input("\nPress Enter...")

        elif choice == '3':
            garage.active_car.reset_service("tires")
            garage.save_fleet()
            input("\nPress Enter...")

        elif choice == '4':
            garage.active_car.edit_data()
            garage.save_fleet()
            input("\nPress Enter...")

        elif choice == '5':
            print("\n --- GARAGE FLEET MANAGER ---")
            for idx, car in enumerate(garage.cars):
                active_flag = " [ACTIVE]" if idx == garage.active_index else ""
                print(f"   [{idx}] {car.brand} {car.model} ({car.engine_type}) - {car.mileage:,} km {active_flag}")
            print("\nActions: [S]witch Active | [A]dd New Car | [B]an/Remove Car | [R]eturn")
            sub_choice = input("Select Action (S/A/B/R): ").strip().lower()

            if sub_choice == 's':
                idx = get_valid_integer("Enter vehicle index to activate: ")
                garage.switch_car(idx)
                input("\nPress Enter...")
            elif sub_choice == 'a':
                print("\n[*] Register New Vehicle:")
                nb = input(" Brand : ").strip()
                nm = input(" Model : ").strip()
                ne = input(" Engine : ").strip()
                km = get_valid_integer("Mileage : ")
                garage.add_car(Car(nb, nm, ne, km))
                input("\nPress Enter...")
            elif sub_choice == 'b':
                idx = get_valid_integer("Enter vehicle index to BAN/REMOVE: ")
                garage.ban_car(idx)
                input("\nPress Enter...")

        elif choice == '6':
            garage.show_maintenance_dashboard()
            input("\nPress Enter...")

        elif choice == '7':
            print("\n[AI JSON Context Preview]:")
            print(garage.get_fleet_llm_context())
            input("\nPress Enter...")

        elif choice == '8':
            print("\n" + "*"*40)
            print(" Safely powering down. Goodbye!")
            print("*"*40)
            input("\n Press Enter to close the terminal..")
            break



== Smart Diagnostics Initialized ==




═══════════════════════════════════════════════════════
   ACTIVE VEHICLE: Toyota RAV4
  • Engine : Hybrid | Mileage: 51,000 km
  • Status : Oil [CHANGE_REQUIRED (51,000 km driven)] | Tires [CHECK_TREAD (51,000 km driven)]
═══════════════════════════════════════════════════════

 GARAGE MANAGMENT SYSTEM V2.8 (Fleet: 2)

[Menu Options]:
 1. Add Trip         
 2. Reset Oil        
 3. Reset Tires      
 4. Edit Vehicle     
 5. Garage Fleet Manager
 6. Fleet Health Dashboard (NEW)
 7. View AI Context  
 8. Exit             

Select (1-8): 2

  [Service] Oil reset at 51,000 km!
   [Storage] Saved to 'fleet_data.json'. (Click ' Refresh' in Colab if hidden)

Press Enter...



═══════════════════════════════════════════════════════
   ACTIVE VEHICLE: Toyota RAV4
  • Engine : Hybrid | Mileage: 51,000 km
  • Status : Oil [OK (10,000 km left)] | Tires [CHECK_TREAD (51,000 km driven)]
═══════════════════════════════════════════════════════

 GARAGE MANAG